In [1]:
from google.colab import drive
# Lệnh này sẽ hiện ra popup yêu cầu bạn cấp quyền truy cập Drive
drive.mount('/content/drive', force_remount=True)
print("✅ Đã kết nối Google Drive thành công!")

Mounted at /content/drive
✅ Đã kết nối Google Drive thành công!


In [4]:
!pip install kagglehub -q

import kagglehub
import os

print("🚀 Đang tải bộ dữ liệu CrowdHuman gốc (Có thể mất 2-3 phút)...")
dataset_path = kagglehub.dataset_download("leducnhuan/crowdhuman")

print("✅ Đã tải xong! Dữ liệu đang nằm tại:", dataset_path)

🚀 Đang tải bộ dữ liệu CrowdHuman gốc (Có thể mất 2-3 phút)...


100%|██████████| 10.1G/10.1G [09:42<00:00, 18.7MB/s]

Extracting files...


✅ Đã tải xong! Dữ liệu đang nằm tại: /root/.cache/kagglehub/datasets/leducnhuan/crowdhuman/versions/1


In [5]:
import json
import os
import cv2
import glob
from tqdm import tqdm

print("🚀 Khởi động Máy lọc Dữ liệu CrowdHuman (Bản Auto-Detect Đường Dẫn)...")

# --- BƯỚC 1: TỰ ĐỘNG DÒ TÌM ĐƯỜNG DẪN GỐC ---
CACHE_DIR = '/root/.cache/kagglehub/datasets'

print(f"🔍 Đang quét radar tìm file gốc trong {CACHE_DIR}...")
train_odgt_list = glob.glob(f'{CACHE_DIR}/**/annotation_train.odgt', recursive=True)
val_odgt_list = glob.glob(f'{CACHE_DIR}/**/annotation_val.odgt', recursive=True)

if not train_odgt_list or not val_odgt_list:
    raise FileNotFoundError("❌ Không tìm thấy file .odgt! Hãy chắc chắn bạn đã chạy lệnh tải kagglehub thành công.")

TRAIN_ODGT = train_odgt_list[0]
VAL_ODGT = val_odgt_list[0]
DATASET_ROOT = os.path.dirname(TRAIN_ODGT) # Thư mục gốc chứa dataset

print(f"✅ Đã chốt vị trí Train: {TRAIN_ODGT}")
print(f"✅ Đã chốt vị trí Val:   {VAL_ODGT}")

# Hàm định vị ảnh siêu việt (Xử lý trường hợp Kaggle chia nhiều thư mục ảnh)
def locate_image(img_id, root_dir):
    paths_to_check = [
        os.path.join(root_dir, 'Images', f"{img_id}.jpg"),
        os.path.join(root_dir, 'Images_val', f"{img_id}.jpg"),
        os.path.join(root_dir, f"{img_id}.jpg")
    ]
    for p in paths_to_check:
        if os.path.exists(p):
            return p
    return None

# --- BƯỚC 2: CẤU HÌNH ĐẦU RA ---
DATASETS = [
    {'name': 'Train', 'odgt': TRAIN_ODGT, 'img_dir': '/content/CrowdHuman_YOLO/images/train', 'lbl_dir': '/content/CrowdHuman_YOLO/labels/train'},
    {'name': 'Validation', 'odgt': VAL_ODGT, 'img_dir': '/content/CrowdHuman_YOLO/images/val', 'lbl_dir': '/content/CrowdHuman_YOLO/labels/val'}
]

# --- BƯỚC 3: TIẾN HÀNH LỌC VÀ CHUYỂN ĐỔI ---
for split in DATASETS:
    print(f"\n⏳ Đang xử lý tập {split['name']}...")
    os.makedirs(split['img_dir'], exist_ok=True)
    os.makedirs(split['lbl_dir'], exist_ok=True)

    with open(split['odgt'], 'r') as f:
        lines = f.readlines()

    count_heads = 0
    count_images = 0

    for line in tqdm(lines, desc=f"Tiến độ {split['name']}"):
        data = json.loads(line)
        image_id = data['ID']

        # Tự động dò tìm bức ảnh này đang ở thư mục nào
        img_path = locate_image(image_id, DATASET_ROOT)

        if not img_path:
            continue

        img = cv2.imread(img_path)
        if img is None: continue
        H, W, _ = img.shape

        valid_boxes = []

        for box in data['gtboxes']:
            if box['tag'] == 'person' and 'hbox' in box:
                x, y, w, h = box['hbox']
                if w <= 0 or h <= 0: continue

                x_center = max(0.0, min(1.0, (x + w / 2.0) / W))
                y_center = max(0.0, min(1.0, (y + h / 2.0) / H))
                w_norm = max(0.0, min(1.0, w / W))
                h_norm = max(0.0, min(1.0, h / H))

                valid_boxes.append(f"0 {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")
                count_heads += 1

        if valid_boxes:
            label_path = os.path.join(split['lbl_dir'], f"{image_id}.txt")
            with open(label_path, 'w') as f_out:
                f_out.write("\n".join(valid_boxes))

            target_img_path = os.path.join(split['img_dir'], f"{image_id}.jpg")
            if not os.path.exists(target_img_path):
                os.symlink(img_path, target_img_path)

            count_images += 1

    print(f"✅ Xong tập {split['name']}: {count_images} ảnh và {count_heads} cái đầu.")

# --- BƯỚC 4: TẠO FILE DATA.YAML ---
print("\n📝 Đang đóng gói file data.yaml...")
import yaml
data_yaml = {
    'train': '/content/CrowdHuman_YOLO/images/train',
    'val': '/content/CrowdHuman_YOLO/images/val',
    'nc': 1,
    'names': ['head']
}
yaml_path = '/content/CrowdHuman_YOLO/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f"✅ Hoàn tất! File cấu hình đã sẵn sàng tại: {yaml_path}")
print("🚀 BỘ DỮ LIỆU ĐÃ SẴN SÀNG CHO GIAI ĐOẠN 2!")

🚀 Khởi động Máy lọc Dữ liệu CrowdHuman (Bản Auto-Detect Đường Dẫn)...
🔍 Đang quét radar tìm file gốc trong /root/.cache/kagglehub/datasets...
✅ Đã chốt vị trí Train: /root/.cache/kagglehub/datasets/leducnhuan/crowdhuman/versions/1/CrowdHuman/annotation_train.odgt
✅ Đã chốt vị trí Val:   /root/.cache/kagglehub/datasets/leducnhuan/crowdhuman/versions/1/CrowdHuman/annotation_val.odgt

⏳ Đang xử lý tập Train...


Tiến độ Train: 100%|██████████| 15000/15000 [04:26<00:00, 56.28it/s]


✅ Xong tập Train: 15000 ảnh và 352983 cái đầu.

⏳ Đang xử lý tập Validation...


Tiến độ Validation: 100%|██████████| 4370/4370 [01:18<00:00, 55.91it/s]

✅ Xong tập Validation: 4370 ảnh và 103115 cái đầu.

📝 Đang đóng gói file data.yaml...
✅ Hoàn tất! File cấu hình đã sẵn sàng tại: /content/CrowdHuman_YOLO/data.yaml
🚀 BỘ DỮ LIỆU ĐÃ SẴN SÀNG CHO GIAI ĐOẠN 2!


In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO

print("🚀 Khởi động Giai đoạn 2: Huấn luyện kiến trúc SAHI-Compatible trên CrowdHuman...")

# 1. Nạp 'bộ não' SCUT-HEAD làm điểm xuất phát (Transfer Learning)
# Lưu ý: Thay đường dẫn này bằng đường dẫn đến file best.pt mạnh nhất của bạn
MODEL_PATH = '/content/drive/MyDrive/SCUT_HEAD_RUNS/yolo11s_ultimate_head-5/weights/best.pt'
model = YOLO(MODEL_PATH)

# 2. Cấu hình Training
results = model.train(
    data='/content/CrowdHuman_YOLO/data.yaml', # File bản đồ ta vừa tạo ở Giai đoạn 1
    epochs=50,          # 50 vòng là đủ để thẩm thấu 15.000 ảnh

    # --- CẤU HÌNH KIẾN TRÚC TƯƠNG THÍCH SAHI ---
    imgsz=640,          # Kích thước chuẩn để đồng bộ với lưới cắt của SAHI sau này
    batch=16,           # Với imgsz=640, Colab T4 GPU có thể gánh được batch=16 thoải mái
    rect=False,         # Ép đưa ảnh về hình vuông (Square Padding) giống hệt lúc SAHI cắt ảnh

    # --- BỘ TỐI ƯU HÓA (Dành riêng cho dữ liệu khổng lồ) ---
    optimizer='auto',   # Với tập data cực lớn, thuật toán Auto của YOLO lại hoạt động rất tốt
    cos_lr=True,        # Giảm tốc độ học mượt mà chống sốc
    patience=10,        # Early Stopping: Tự động dừng nếu sau 10 vòng mà mAP không tăng

    # --- TĂNG CƯỜNG DỮ LIỆU ĐÁM ĐÔNG ---
    mosaic=1.0,         # Bật tối đa Mosaic. Trộn 4 ảnh CrowdHuman vào nhau = Đám đông siêu cấp
    mixup=0.1,          # Trộn nhẹ các bức ảnh để tăng tính tổng quát hóa

    # --- LƯU TRỮ ---
    project='/content/drive/MyDrive/CrowdHuman_SAHI_Training', # Lưu thẳng lên Drive
    name='run_1'
)

print("\n🎉 Tuyệt vời! Mô hình đang được rèn luyện. Việc của bạn bây giờ là đi pha một tách cà phê thật ngon!")

🚀 Khởi động Giai đoạn 2: Huấn luyện kiến trúc SAHI-Compatible trên CrowdHuman...
Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/CrowdHuman_YOLO/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=/content/drive/MyDrive/SCUT_HEAD_RUNS/yolo11s_ultimate_head-5/weights/best.pt, 

In [7]:
# Đừng quên cài đặt ultralytics nếu Colab bị reset hoàn toàn
!pip install ultralytics -q

from ultralytics import YOLO

print("🚀 Kích hoạt tính năng Hồi sinh (Resume Training)...")

# 1. TRỎ VÀO FILE 'last.pt' CỦA LẦN TRAIN TRƯỚC
# File last.pt chứa toàn bộ trọng số, optimizer state, và epoch hiện tại
# Chú ý: Đây là file last.pt, KHÔNG PHẢI best.pt
LAST_MODEL_PATH = '/content/drive/MyDrive/CrowdHuman_SAHI_Training/run_1/weights/last.pt'

# Nạp file last.pt vào mô hình
model = YOLO(LAST_MODEL_PATH)

# 2. Gọi lệnh Train với tham số Resume = True
print(f"Đang tiếp tục huấn luyện từ checkpoint cuối cùng...")
results = model.train(resume=True)

print("\n🎉 Tuyệt vời! Mô hình đã sống lại và tiếp tục chặng đường.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 60.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 Kích hoạt tính năng Hồi sinh (Resume Training)...
Đang tiếp tục huấn luyện từ checkpoint cuối cùng...
Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/CrowdHuman_YOLO/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, 

In [ ]:
!pip install ultralytics -q
!pip install sahi -q

import cv2
import os
from tqdm import tqdm
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

print("🚀 Khởi động Giai đoạn 3: Trích xuất Video với lưới tinh thể SAHI (Custom Pipeline)...")

# 1. Cấu hình đường dẫn
MODEL_PATH = '/content/drive/MyDrive/CrowdHuman_SAHI_Training/run_1/weights/best.pt'
VIDEO_PATH = '/content/TownCentre_1min.mp4'
OUTPUT_PATH = '/content/TownCentre_SAHI_Result.mp4'

if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError("❌ Không tìm thấy video TownCentre_1min.mp4. Bạn hãy upload nó lên lại Colab nhé!")

# 2. Nạp 'bộ não' YOLO vào lõi SAHI
print("🧠 Đang nạp trọng số YOLO vào SAHI...")
detection_model = AutoDetectionModel.from_pretrained(
    model_type='yolov8',
    model_path=MODEL_PATH,
    confidence_threshold=0.3, # Độ tự tin 30%
    device="cuda:0",          # Chạy bằng GPU
)

# 3. Khởi tạo bộ đọc và ghi Video (OpenCV)
cap = cv2.VideoCapture(VIDEO_PATH)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

print(f"🎬 Bắt đầu rà quét {total_frames} khung hình (Sẽ mất khoảng vài phút)...")

# 4. Vòng lặp xử lý từng Frame
for i in tqdm(range(total_frames), desc="Tiến độ phân tích"):
    ret, frame = cap.read()
    if not ret:
        break

    # SAHI yêu cầu ảnh hệ màu RGB, OpenCV mặc định là BGR
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # DÙNG KÍNH HIỂN VI SAHI ĐỂ QUÉT TỪNG Ô LƯỚI
    result = get_sliced_prediction(
        rgb_frame,
        detection_model,
        slice_height=640,
        slice_width=640,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        verbose=0 # Tắt log rác
    )

    # Trích xuất tọa độ và vẽ Box trực tiếp lên Frame
    for obj in result.object_prediction_list:
        bbox = obj.bbox
        x1, y1, x2, y2 = int(bbox.minx), int(bbox.miny), int(bbox.maxx), int(bbox.maxy)
        conf = obj.score.value

        # Vẽ hộp màu xanh lá cây
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        # Ghi tỷ lệ % tự tin
        cv2.putText(frame, f"{conf:.2f}", (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Ghi Frame đã vẽ vào Video mới
    out.write(frame)

# Giải phóng bộ nhớ
cap.release()
out.release()

print(f"\n🎉 QUÁ TRÌNH HOÀN TẤT! Video kết quả đã được lưu tại: {OUTPUT_PATH}")
print("👉 Hãy nhấp vào biểu tượng thư mục ở thanh bên trái Colab, tải file TownCentre_SAHI_Result.mp4 về máy và chiêm ngưỡng!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.2 MB/s eta 0:00:00
🚀 Khởi động Giai đoạn 3: Trích xuất Video với lưới tinh thể SAHI (Custom Pipeline)...
🧠 Đang nạp trọng số YOLO vào SAHI...
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🎬 Bắt đầu rà quét 1501 khung hình (Sẽ mất khoảng vài phút)...


Tiến độ phân tích:  30%|██▉       | 447/1501 [38:02<1:28:55,  5.06s/it]